# Train Crowd Forecast Model

Notebook nay duoc viet de chay tren Google Colab. Muc tieu la train baseline crowd forecast model dau tien cho Eco-Travel Coordinator theo huong de lam truoc va de mo rong nhieu thanh pho sau nay.

## 1. Cai dat thu vien

In [ ]:
!pip install -q pandas scikit-learn joblib xgboost

## 2. Cau hinh duong dan

Cach de nhat la upload ca thu muc `eco_travel_coordinator` len `/content/eco_travel_coordinator`.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/eco_travel_coordinator')
SAMPLE_DIR = PROJECT_DIR / 'data' / 'sample'
TRAINING_DIR = PROJECT_DIR / 'data' / 'training'
MODELS_DIR = PROJECT_DIR / 'models_artifacts'

ATTRACTIONS_FILE = SAMPLE_DIR / 'attractions.csv'
CROWD_HISTORY_FILE = SAMPLE_DIR / 'crowd_history.csv'
TRAINING_FILE = TRAINING_DIR / 'crowd_training_dataset.csv'
CLEANED_OUTPUT = TRAINING_DIR / 'cleaned_training_data.csv'
MODEL_OUTPUT = MODELS_DIR / 'crowd_forecast_model.pkl'
PREPROCESSOR_OUTPUT = MODELS_DIR / 'crowd_preprocessor.pkl'
METADATA_OUTPUT = MODELS_DIR / 'crowd_model_metadata.json'
DEFAULT_CITY = 'Unknown City'

TRAINING_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('ATTRACTIONS_FILE exists:', ATTRACTIONS_FILE.exists())
print('CROWD_HISTORY_FILE exists:', CROWD_HISTORY_FILE.exists())

## 3. Helper functions de tu chuan bi dataset

Neu chua co file training, notebook se tu build tu `attractions.csv` va `crowd_history.csv`.

In [ ]:
import json
import shutil
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ATTRACTION_REQUIRED = {
    'attraction_id', 'name', 'category', 'estimated_capacity', 'popularity_score', 'indoor_outdoor'
}
HISTORY_REQUIRED = {
    'timestamp', 'attraction_id', 'crowd_score', 'weather', 'holiday_flag', 'event_flag', 'day_of_week', 'hour'
}

def validate_columns(df, required_columns, dataset_name):
    missing = sorted(required_columns - set(df.columns))
    if missing:
        raise ValueError(f'{dataset_name} thieu cot bat buoc: {missing}')

def build_training_dataframe(attractions_path, history_path, default_city='Unknown City'):
    attractions = pd.read_csv(attractions_path)
    history = pd.read_csv(history_path)

    validate_columns(attractions, ATTRACTION_REQUIRED, 'attractions.csv')
    validate_columns(history, HISTORY_REQUIRED, 'crowd_history.csv')

    if 'city' not in attractions.columns:
        attractions['city'] = default_city
    attractions['city'] = attractions['city'].fillna(default_city).astype(str).str.strip()
    attractions.loc[attractions['city'] == '', 'city'] = default_city

    history['timestamp'] = pd.to_datetime(history['timestamp'], errors='coerce')
    history = history.dropna(subset=['timestamp', 'attraction_id', 'crowd_score']).copy()
    history = history.sort_values(['attraction_id', 'timestamp']).reset_index(drop=True)

    if 'day_of_week' not in history.columns:
        history['day_of_week'] = history['timestamp'].dt.day_name()
    if 'hour' not in history.columns:
        history['hour'] = history['timestamp'].dt.hour

    history['historical_crowd_score'] = history.groupby('attraction_id')['crowd_score'].shift(1)
    history['historical_crowd_score'] = history['historical_crowd_score'].fillna(history['crowd_score'])

    merged = history.merge(
        attractions[['attraction_id', 'name', 'city', 'category', 'estimated_capacity', 'popularity_score', 'indoor_outdoor']],
        how='left',
        on='attraction_id',
    )

    merged['city'] = merged['city'].fillna(default_city)
    merged['category'] = merged['category'].fillna('khong_ro')
    merged['indoor_outdoor'] = merged['indoor_outdoor'].fillna('khong_ro')
    merged['weather'] = merged['weather'].fillna('khong_ro')
    merged['day_of_week'] = merged['day_of_week'].fillna(merged['timestamp'].dt.day_name())

    numeric_defaults = {
        'popularity_score': 0,
        'estimated_capacity': 0,
        'hour': 0,
        'holiday_flag': 0,
        'event_flag': 0,
        'historical_crowd_score': 0,
        'temperature': 0,
        'rain_flag': 0,
        'crowd_score': 0,
    }
    for column_name, default_value in numeric_defaults.items():
        if column_name not in merged.columns:
            merged[column_name] = default_value
        merged[column_name] = pd.to_numeric(merged[column_name], errors='coerce').fillna(default_value)

    output_columns = [
        'timestamp', 'attraction_id', 'name', 'city', 'category', 'popularity_score',
        'estimated_capacity', 'hour', 'day_of_week', 'weather', 'holiday_flag',
        'event_flag', 'historical_crowd_score', 'indoor_outdoor', 'temperature',
        'rain_flag', 'crowd_score',
    ]
    dataset = merged[output_columns].copy()
    dataset['crowd_score'] = dataset['crowd_score'].clip(0, 100)
    dataset['historical_crowd_score'] = dataset['historical_crowd_score'].clip(0, 100)
    dataset['hour'] = dataset['hour'].astype(int).clip(0, 23)
    dataset['holiday_flag'] = dataset['holiday_flag'].astype(int).clip(0, 1)
    dataset['event_flag'] = dataset['event_flag'].astype(int).clip(0, 1)
    dataset['rain_flag'] = dataset['rain_flag'].astype(int).clip(0, 1)
    return dataset


## 4. Load hoac tao training dataset

In [ ]:
if TRAINING_FILE.exists():
    df = pd.read_csv(TRAINING_FILE)
    print('Da tim thay training file san:', TRAINING_FILE)
else:
    df = build_training_dataframe(ATTRACTIONS_FILE, CROWD_HISTORY_FILE, DEFAULT_CITY)
    df.to_csv(TRAINING_FILE, index=False)
    print('Da tao training file moi tu CSV goc.')

df.to_csv(CLEANED_OUTPUT, index=False)
print('Shape:', df.shape)
df.head()

## 5. EDA co ban

In [ ]:
print('So dong:', len(df))
print('So attraction:', df['attraction_id'].nunique())
print('So city:', df['city'].nunique())
print('Danh sach city:', sorted(df['city'].dropna().astype(str).unique().tolist()))
print('\nThong ke crowd_score:')
print(df['crowd_score'].describe())
print('\nTop category:')
print(df['category'].value_counts().head(10))

## 6. Chon feature va xu ly missing values

In [ ]:
target_column = 'crowd_score'
feature_columns = [
    'attraction_id',
    'city',
    'category',
    'popularity_score',
    'estimated_capacity',
    'hour',
    'day_of_week',
    'weather',
    'holiday_flag',
    'event_flag',
    'historical_crowd_score',
    'indoor_outdoor',
    'temperature',
    'rain_flag',
]
categorical_features = ['attraction_id', 'city', 'category', 'day_of_week', 'weather', 'indoor_outdoor']
numeric_features = [column for column in feature_columns if column not in categorical_features]

model_df = df.copy()
model_df = model_df.dropna(subset=[target_column]).sort_values('timestamp').reset_index(drop=True)

X = model_df[feature_columns]
y = model_df[target_column]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
            ]),
            numeric_features,
        ),
    ]
)

print('Feature columns:', feature_columns)

## 7. Train/validation split

Mac dinh uu tien time-based split neu co timestamp. Neu can, ban co the doi sang random split sau.

In [ ]:
split_index = int(len(model_df) * 0.8)
split_index = max(split_index, 1)

X_train = X.iloc[:split_index].copy()
X_valid = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_valid = y.iloc[split_index:].copy()

if len(X_valid) == 0:
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

print('Train size:', len(X_train))
print('Valid size:', len(X_valid))

## 8. Train baseline RandomForestRegressor

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    (
        'model',
        RandomForestRegressor(
            n_estimators=300,
            max_depth=16,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

rf_pipeline.fit(X_train, y_train)
rf_pred = np.clip(rf_pipeline.predict(X_valid), 0, 100)

rf_metrics = {
    'model_name': 'random_forest',
    'mae': float(mean_absolute_error(y_valid, rf_pred)),
    'rmse': float(mean_squared_error(y_valid, rf_pred, squared=False)),
    'r2': float(r2_score(y_valid, rf_pred)),
}
rf_metrics

## 9. Thu XGBoost neu muon so sanh

Baseline van la RandomForest. Cell nay chi de so sanh neu ban muon.

In [ ]:
best_pipeline = rf_pipeline
best_metrics = rf_metrics.copy()

try:
    from xgboost import XGBRegressor

    xgb_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        (
            'model',
            XGBRegressor(
                n_estimators=400,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                objective='reg:squarederror',
                random_state=42,
            ),
        ),
    ])

    xgb_pipeline.fit(X_train, y_train)
    xgb_pred = np.clip(xgb_pipeline.predict(X_valid), 0, 100)
    xgb_metrics = {
        'model_name': 'xgboost',
        'mae': float(mean_absolute_error(y_valid, xgb_pred)),
        'rmse': float(mean_squared_error(y_valid, xgb_pred, squared=False)),
        'r2': float(r2_score(y_valid, xgb_pred)),
    }
    print('XGBoost metrics:', xgb_metrics)

    if xgb_metrics['rmse'] < best_metrics['rmse']:
        best_pipeline = xgb_pipeline
        best_metrics = xgb_metrics
except Exception as exc:
    print('Bo qua XGBoost:', exc)

best_metrics

## 10. Export artifacts

Notebook se luu full sklearn pipeline, preprocessor rieng, metadata va cleaned training data.

In [ ]:
joblib.dump(best_pipeline, MODEL_OUTPUT)
joblib.dump(best_pipeline.named_steps['preprocessor'], PREPROCESSOR_OUTPUT)
model_df.to_csv(CLEANED_OUTPUT, index=False)

metadata = {
    'model_name': best_metrics['model_name'],
    'artifact_type': 'sklearn_pipeline',
    'feature_columns': feature_columns,
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'target_column': target_column,
    'metrics': best_metrics,
    'training_rows': int(len(model_df)),
    'validation_rows': int(len(X_valid)),
    'city_count': int(model_df['city'].nunique()),
    'attraction_count': int(model_df['attraction_id'].nunique()),
}

METADATA_OUTPUT.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved model:', MODEL_OUTPUT)
print('Saved preprocessor:', PREPROCESSOR_OUTPUT)
print('Saved metadata:', METADATA_OUTPUT)
print('Saved cleaned data:', CLEANED_OUTPUT)

## 11. Nen file export de tai xuong de hon

In [ ]:
archive_path = shutil.make_archive(str(MODELS_DIR / 'crowd_model_exports'), 'zip', root_dir=PROJECT_DIR, base_dir='models_artifacts')
print('Archive:', archive_path)

try:
    from google.colab import files
    # files.download(archive_path)
    print('Neu muon tai xuong ngay, bo comment dong files.download(archive_path).')
except Exception:
    print('Khong phai moi truong Colab, bo qua buoc download.')

## 12. Huong mo rong multi-city

- Khi mo rong du lieu, chi can bo sung cot `city` thuc vao attractions.
- Neu co nhieu tinh/thanh, co the them `district`, `tourism_cluster`, `season`, `special_event_name`.
- Khi dataset lon hon, co the doi sang validation theo city hoac time series split.
- Artifact hien tai dat trong `models_artifacts/` de sau nay noi vao app desktop.